In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
import base64

In [2]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [3]:
vision_llm = ChatOllama(model="gemma3:12b", temperature=0., num_predict=4096)

In [4]:
def image_analyzer(image_path: str) -> str:
    encoded_image = encode_image(image_path)
    message = HumanMessage(
        content=[
            {"type": "text", "text": "analyze the picture, then list ALL objects in JSON format."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{encoded_image}"
                },
            },
        ]
    )
    response = vision_llm.invoke([message])
    return response.content

<img src="imgs/img.png"/>

In [5]:
print(image_analyzer("imgs/img.png"))

Okay, this is a challenging one! It's a densely packed image with a huge variety of objects. I'll do my best to identify as many as possible and present them in JSON format.  Given the density and overlapping nature of the image, some identifications might be approximate or based on partial visibility.

**Important Notes:**

*   **Approximation:** This list is based on visual interpretation and might not be exhaustive. There are likely many smaller or obscured items I'm missing.
*   **Grouping:** I'm grouping similar items to keep the list manageable.
*   **Variations:**  I'm not listing every single variation of a type of object (e.g., different colors of screws).
*   **Perspective:** The perspective and overlapping make precise identification difficult.

Here's the JSON representation of the objects I can identify:

```json
{
  "objects": [
    {
      "category": "Vehicles",
      "items": [
        {"type": "Taxi", "color": "Yellow"},
        {"type": "Car", "color": "Silver"},
   

In [6]:
prompt_template = PromptTemplate(
    input_variables=["image_description", "marketing_claim", "marketing_description"],
    template=(
        """Agisci come un consulente di marketing con 10 anni di esperienza.
Hai ricevuto la seguente descrizione di una campagna:
Claim: {marketing_claim}
Descrizione: {marketing_description}

Hai anche ricevuto la seguente descrizione dell'immagine da analizzare:
{image_description}

In base a claim e descrizione, suggerisci come migliorare l'immagine per renderla più efficace per la campagna marketing.
Fornisci consigli specifici e concreti (es. aggiungere elementi grafici, cambiare colori, inserire testo, modificare l'angolazione, inserire persone o contesto).
Non inserire preamboli e conclusioni, limitati a descrivere dei consigli specifici e concreti per migliorare l'immagine.
"""
    )
)

In [7]:
# Catena per generare i suggerimenti
suggestions_chain = prompt_template | vision_llm

<img src="imgs/caffe.png">

In [8]:
# Ora simuliamo l'input
image_path = "imgs/caffe.png"  # Immagine di input
marketing_claim = "Il miglior caffè artigianale per iniziare la giornata"
marketing_description = "Una miscela premium di caffè biologico, appena tostato, ideale per chi ama l'aroma intenso e il gusto pieno. La campagna punta a trasmettere l'idea di un prodotto esclusivo, raffinato e adatto a chi vuole distinguersi."

In [9]:
# Ora generiamo i suggerimenti finali
suggestions = suggestions_chain.invoke({
    "image_description": image_analyzer(image_path),
    "marketing_claim": marketing_claim,
    "marketing_description": marketing_description
})

print("Suggerimenti per migliorare l'immagine:")
print(suggestions.content)

Suggerimenti per migliorare l'immagine:
*   **Eliminare o sfocare il Coffee Roaster:** La presenza della macchina da torrefazione distrae dall'idea di artigianalità e freschezza. Suggerirebbe un processo industriale, in contrasto con il claim di caffè artigianale.
*   **Migliorare l'illuminazione:** L'immagine sembra un po' spenta. Aumentare la luminosità e utilizzare una luce più calda per esaltare i colori dei chicchi e creare un'atmosfera più invitante.
*   **Cambiare l'abbigliamento della mano:** La camicia a quadri rossa e nera è troppo casual e non si allinea con l'idea di un prodotto esclusivo e raffinato. Sostituirla con una camicia più elegante, magari in colori neutri come il bianco, il grigio chiaro o il beige.
*   **Modificare l'angolazione della mano:** L'angolazione attuale della mano che tiene i chicchi non è particolarmente interessante. Provare un'angolazione che mostri meglio la consistenza e la qualità dei chicchi, magari con la luce che li illumina lateralmente.
*  